# Notebook 6: EviDTI Model Training and Evaluation (Davis)

**Purpose:** Train and evaluate the reproduced EviDTI model on the Davis dataset.

**Inputs:** Pre-extracted Davis features from `/content/davis/`:
- `davis_t_feature.npy` — protein features
- `davis_d_2D_features.npy` — drug 2D features
- `davis_d_3d_feature.npy` — drug 3D features
- `davis_total_cid_unid.csv` — drug-target interaction pairs

**Output:** Test set evaluation metrics and `EviDTI_Davis_Test_Predictions_Detailed.csv`
with per-sample probability and uncertainty scores.

**Reproducibility:** `np.random.seed(3)` is applied, consistent with the original codebase.

**Note:** Davis uses a fixed `lam=0.1` (no KL annealing), matching the original paper.

## Step 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Extract Project Files and Davis Dataset

In [ ]:
# Extract project code
!unzip -q /content/drive/MyDrive/Capstone_Project/EviDTI-main.zip -d /content/

# Extract Davis dataset
!unzip -q /content/drive/MyDrive/Capstone_Project/davis.zip -d /content/

## Step 2: Install Dependencies

In [ ]:
# Install PyTorch Geometric
!pip install torch_geometric

# Install yacs (YAML config reader)
!pip install yacs

# Install TensorBoard
!pip install tensorboard
!pip install rdkit

## Step 3: Apply Source Code Patches

**Patch 1 (`main_davis.py`):** Update data paths to Davis dataset location, attach test
loader for final evaluation, and add CSV export of per-sample predictions.

**Patch 2 (`davis_solver.py`):** Fix `loss.mean()`, label `float32`, and belief mass formula.

**Patch 3 (`davis_model.py`):** Fix PyG Batch collation of bag sub-graphs, 1D edge
attribute guard, and Light Attention mask alignment.

**Patch 4 (`function.py`):** Fix PyTorch 2.6 `weights_only` policy and `davis.pt` filename.

**Patch 5 (`configs/davis.yaml`):** Set `MAX_EPOCH` to 10.

In [ ]:
import os
import re

print("Resetting codebase to original state...")
# Force re-extract to reset codebase
os.system("unzip -o -q /content/drive/MyDrive/EviDTI-main.zip -d /content/")
os.system("rm -rf /content/EviDTI-main/run /content/EviDTI-main/runs")  # Clear logs


print("Applying all patches...")


# Patch 1: main_davis.py — update data paths + attach test set + export detailed CSV
f_main = '/content/EviDTI-main/main_davis.py'
with open(f_main, 'r', encoding='utf-8') as f: c = f.read()

new_data_code = """    base_path = '/content/davis/'
    data_list = davis_graph_data(root=base_path,
                                t_1D_path=base_path + 'davis_t_feature.npy',
                                d_2D_path=base_path + 'davis_d_2D_features.npy',
                                d_3D_path=base_path + 'davis_d_3d_feature.npy',
                                label_file=base_path + 'davis_total_cid_unid.csv',
                                metadata_path=base_path + 'davis_total_cid_unid.csv')
    np.random.seed(3)"""
c = re.sub(r'[ \t]*data_list\s*=\s*.*?np\.random\.seed\(3\)', new_data_code, c, flags=re.DOTALL)

export_code = """
    print('\\nExtracting test set predictions and uncertainty scores...')
    _, _, test_results, test_c, test_var, test_prob, test_ev, _ = solver.predict_val(test_loader, cfg.SOLVER.MAX_EPOCH)
    import numpy as np, pandas as pd
    alpha = test_ev + 1.0
    S = np.sum(alpha, axis=1)
    df_detailed = pd.DataFrame({
        'True_Label': test_results[:, 1], 'Pred_Class': test_results[:, 0],
        'Pred_Prob_Class_1': test_prob, 'Confidence': np.max(alpha / S[:, None], axis=1),
        'Uncertainty': 2.0 / S, 'Evidence_0': test_ev[:, 0], 'Evidence_1': test_ev[:, 1]
    })
    df_detailed.to_csv('Davis_Phase2_Predictions_Detailed.csv', index=False)
    print('Davis_Phase2_Predictions_Detailed.csv saved.')
"""
c = c.replace("solver.train(train_loader, val_loader)", "solver.train(train_loader, val_loader, eval_data=test_loader)\n" + export_code)
with open(f_main, 'w', encoding='utf-8') as f: f.write(c)

# Patch 2: davis_solver.py — fix loss.mean(), float32, belief mass formula, and epoch logging
f_sol = '/content/EviDTI-main/davis_solver.py'
with open(f_sol, 'r', encoding='utf-8') as f: s = f.read()
s = s.replace("loss = loss.sum()", "loss = loss.mean()")
s = s.replace('label.to(torch.float16)', 'label.to(torch.float32)')
s = s.replace('bk = alphas -1 / np.sum(alphas, axis=-1)', 'bk = (alphas - 1) / (np.sum(alphas, axis=-1, keepdims=True) + 1e-9)')

print_rep = """
        print('[Epoch %d] Train Loss: %.4f | Val Acc: %.2f%% | Val MCC: %.4f' % (epoch + 1, train_loss, val_acc*100, val_mcc))
        import pandas as pd, os as _os
        save_dir = getattr(self, 'result_dir', getattr(self, 'save_dir', self.writer.log_dir))
        _os.makedirs(save_dir, exist_ok=True)
        if not hasattr(self, 'history_df'): self.history_df = []
        self.history_df.append({'Epoch': epoch + 1, 'Train_Loss': train_loss, 'Val_Acc': val_acc*100, 'Val_MCC': val_mcc})
        pd.DataFrame(self.history_df).to_csv(_os.path.join(save_dir, 'davis_training_log.csv'), index=False)
"""
s = re.sub(r"print\('\[Epoch %d\] val accuracy: %\.4f' % \(epoch \+ 1, val_acc\)\)", print_rep, s)
with open(f_sol, 'w', encoding='utf-8') as f: f.write(s)

# Patch 3: davis_model.py — fix PyG Batch collation, edge attr dimension, mask alignment
f_mod = '/content/EviDTI-main/davis_model.py'
with open(f_mod, 'r', encoding='utf-8') as f: m = f.read()
m = re.sub(r'    def forward\(self, data\):', "    def forward(self, data):\n        if hasattr(data, 'bag') and isinstance(data.bag, list):\n            from torch_geometric.data import Batch\n            data.bag = Batch.from_data_list(data.bag).to(data.x.device)", m, count=1)
m = re.sub(r"([ \t]*)edge_int_feat = edge_attr_list\[:, 0:3\]\.to\('cpu'\)\.to\(torch\.int64\)", r"\1if hasattr(edge_attr_list, 'dim') and edge_attr_list.dim() == 1:\n\1    edge_attr_list = edge_attr_list.view(-1, 4) if edge_attr_list.numel() % 4 == 0 else edge_attr_list.view(-1, 1)\n\1edge_int_feat = edge_attr_list[:, 0:3].to('cpu').to(torch.int64)", m)
m = re.sub(r"([ \t]*)attention = attention\.masked_fill\(mask\[:, None, :\] == False, -1e9\)", r"\1if mask is not None and mask.size(-1) != attention.size(-1):\n\1    import torch.nn.functional as F\n\1    if mask.size(-1) < attention.size(-1):\n\1        mask = F.pad(mask, (0, attention.size(-1) - mask.size(-1)), value=False)\n\1    else:\n\1        mask = mask[:, :attention.size(-1)]\n\1attention = attention.masked_fill(mask[:, None, :] == False, -1e9)", m)
with open(f_mod, 'w', encoding='utf-8') as f: f.write(m)

# Patch 4: function.py — fix PyTorch 2.6 weights_only policy and davis.pt filename
f_func = '/content/EviDTI-main/function.py'
with open(f_func, 'r', encoding='utf-8') as f: fc = f.read()
fc = fc.replace("return ['drugbank.pt']", "return ['davis.pt']")
fc = fc.replace("torch.load(self.processed_paths[0])", "torch.load(self.processed_paths[0], weights_only=False)")
with open(f_func, 'w', encoding='utf-8') as f: f.write(fc)

# Patch 5: configs/davis.yaml — set MAX_EPOCH to 10
f_cfg = '/content/EviDTI-main/configs/davis.yaml'
if os.path.exists(f_cfg):
    with open(f_cfg, 'r', encoding='utf-8') as f: yc = f.read()
    yc = re.sub(r'MAX_EPOCH:\s*\d+', 'MAX_EPOCH: 10', yc)
    with open(f_cfg, 'w', encoding='utf-8') as f: f.write(yc)

print("All patches applied successfully.")

## Step 4: Additional Solver Patches

**Patch 6:** Inject export code for per-sample predictions into `main_davis.py`.

**Patch 7:** Revert `lam` to fixed 0.1 in `davis_solver.py` (Davis does not use KL annealing).

**Patches 8-9:** Add `val_loss` computation and TensorBoard dual-track Loss logging to `davis_solver.py`.

In [ ]:
with open('/content/EviDTI-main/main_davis.py', 'r') as f:
    code = f.read()

export_code = """
    print('\\nExtracting test set predictions and uncertainty scores...')
    _, _, test_p, test_c, test_var, test_prob, test_ev, _, _ = solver.predict_val(test_loader)
    import pandas as pd
    test_results = np.array(test_p)
    test_ev_arr = np.array([e[0] for e in test_ev])
    alpha = test_ev_arr + 1.0
    S = np.sum(alpha, axis=1)
    df_detailed = pd.DataFrame({
        'True_Label': test_results[:, 1],
        'Pred_Class': test_results[:, 0],
        'Pred_Prob_Class_1': test_prob,
        'Uncertainty': 2.0 / S,
    })
    df_detailed.to_csv('Davis_Phase2_Predictions_Detailed.csv', index=False)
    print('Davis_Phase2_Predictions_Detailed.csv saved.')
"""

old = "    solver.train(train_loader, val_loader,eval_data=test_loader)"
new = "    solver.train(train_loader, val_loader,eval_data=test_loader)" + export_code

if old in code:
    code = code.replace(old, new)
    with open('/content/EviDTI-main/main_davis.py', 'w') as f:
        f.write(code)
    print("Patch applied.")
else:
    print("Target code not found, may already be patched.")

In [ ]:
import os

solver_path = '/content/EviDTI-main/davis_solver.py'

# Read solver file
with open(solver_path, 'r', encoding='utf-8') as f:
    s = f.read()

# Revert lam=current_lam back to lam=0.1 (Davis uses fixed lam)
s = s.replace("lam=current_lam", "lam=0.1")

# Remove any residual current_lam definition
s = s.replace("current_lam = min(0.3, (epoch + 1) / 10.0)", "")

# Write back
with open(solver_path, 'w', encoding='utf-8') as f:
    f.write(s)

# Clear Python cache to force reload of the patched file
os.system("rm -rf /content/EviDTI-main/__pycache__")

print("Patch applied: lam reverted to 0.1, current_lam removed.")

In [ ]:
with open('/content/EviDTI-main/davis_solver.py', 'r') as f:
    code = f.read()

# Fix 1: initialise running_val_loss in predict_val
old1 = """    def predict_val(self, data_loader: DataLoader, epoch: int = None):
        preds = []
        labels = []
        t_id = []
        d_id = []
        for i, batch in enumerate(data_loader):"""

new1 = """    def predict_val(self, data_loader: DataLoader, epoch: int = None):
        preds = []
        labels = []
        t_id = []
        d_id = []
        running_val_loss = 0
        for i, batch in enumerate(data_loader):"""

# Fix 2: compute val_loss per batch
old2 = """            batch_preds = batch_preds.tolist()
            label_org = label_org.tolist()"""

new2 = """            label_onehot = torch.nn.functional.one_hot(label_org).to(torch.float32)
            val_loss_batch = self.loss_func(label_onehot, alphas=batch_preds, lam=0.1)
            running_val_loss += val_loss_batch.mean().item()
            batch_preds = batch_preds.tolist()
            label_org = label_org.tolist()"""

# Fix 3: include val_loss in return value
old3 = """        return t_id,d_id,p, c, var,prob_list,ev,bk_list"""
new3 = """        running_val_loss /= len(data_loader)
        return t_id,d_id,p, c, var,prob_list,ev,bk_list,running_val_loss"""

# Fix 4: unpack val_loss in train loop
old4 = """                _,_,val_results,c,var,prob_list,_,_ = self.predict_val(val_loader, epoch + 1)"""
new4 = """                _,_,val_results,c,var,prob_list,_,_,val_loss = self.predict_val(val_loader, epoch + 1)"""

# Fix 5: remove old print statement
old5 = """            print('[Epoch %d] val accuracy: %.4f%% train accuracy: %.4f%%' % (epoch + 1, val_acc, train_acc))"""
new5 = ""

# Fix 6: add val_loss to TensorBoard and update epoch print
old6 = """            self.writer.add_scalars('Acc', {'train': train_acc, 'val': val_acc}, epoch + 1)
            self.writer.add_scalars('MCC', {'train': train_mcc, 'val': val_mcc}, epoch + 1)"""
new6 = """            self.writer.add_scalars('Acc', {'train': train_acc, 'val': val_acc}, epoch + 1)
            self.writer.add_scalars('MCC', {'train': train_mcc, 'val': val_mcc}, epoch + 1)
            self.writer.add_scalars('Loss', {'train': train_loss, 'val': val_loss}, epoch + 1)
            print('[Epoch %d] Train Loss: %.4f | Val Loss: %.4f | Train Acc: %.2f%% | Val Acc: %.4f%% | Val MCC: %.4f' % (epoch + 1, train_loss, val_loss, train_acc, val_acc, val_mcc))"""

replacements = [
    (old1, new1),
    (old2, new2),
    (old3, new3),
    (old4, new4),
    (old5, new5),
    (old6, new6),
]

for i, (old, new) in enumerate(replacements):
    if old in code:
        code = code.replace(old, new)
        print(f"Fix {i+1} applied.")
    else:
        print(f"Fix {i+1}: target not found, may already be patched.")

with open('/content/EviDTI-main/davis_solver.py', 'w') as f:
    f.write(code)

print("davis_solver.py patched.")

In [ ]:
with open('/content/EviDTI-main/davis_solver.py', 'r') as f:
    code = f.read()

old = "            t_id,d_id,p, c, var,prob_list,ev,bk_list = self.predict_val(eval_dataset)"
new = "            t_id,d_id,p, c, var,prob_list,ev,bk_list,_ = self.predict_val(eval_dataset)"

if old in code:
    code = code.replace(old, new)
    with open('/content/EviDTI-main/davis_solver.py', 'w') as f:
        f.write(code)
    print("Patch applied.")
else:
    print("Target code not found, may already be patched.")

## Step 5: Train the Model

In [ ]:
# Clear any previous run artifacts before training
!rm -rf /content/EviDTI-main/runs

# Change to project directory
%cd /content/EviDTI-main

# Run Davis training (10 epochs as set in configs/davis.yaml)
!python main_davis.py --cfg configs/davis.yaml --data davis --split random

## Step 6: Save Training Results to Google Drive

In [ ]:
# Back up training outputs to Google Drive
print("Backing up training results to Google Drive...")

# Define backup path
backup_dir = "/content/drive/MyDrive/EviDTI_Davis_Train_Results_Backup/"

!mkdir -p {backup_dir}




# Back up runs folder (logs and best checkpoint)
!cp -r /content/EviDTI-main/runs {backup_dir} 2>/dev/null || true

# Back up other outputs
!cp -r /content/EviDTI-main/result {backup_dir} 2>/dev/null || true
!cp -r /content/EviDTI-main/model_configs {backup_dir} 2>/dev/null || true

# Back up loose .pt and .csv files
!cp /content/EviDTI-main/*.csv {backup_dir} 2>/dev/null || true
!cp /content/EviDTI-main/*.pt {backup_dir} 2>/dev/null || true



print("Backup complete.")




## Step 7: Evaluate on Test Set

In [ ]:
# EviDTI Davis test set evaluation
import sys
import os
import torch
import numpy as np
from torch_geometric.loader import DataLoader
from torch.utils.data import SubsetRandomSampler

# Fix seed to reproduce the exact same split used during training
np.random.seed(3)

# 1. Set working directory
os.chdir('/content/EviDTI-main')
if '/content/EviDTI-main' not in sys.path:
    sys.path.append('/content/EviDTI-main')

# Import Davis-specific modules
from davis_configs import get_cfg_defaults
from davis_model import LightAttention
from davis_solver import Solver
from function import davis_graph_data

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load config
print("Loading config...")
cfg = get_cfg_defaults()
cfg.merge_from_file('configs/davis.yaml')
cfg.freeze()

# 3. Load dataset and reproduce the same split used during training
print("Loading Davis dataset...")
base_path = '/content/davis/'
data_list = davis_graph_data(root=base_path,
                             t_1D_path=base_path + 'davis_t_feature.npy',
                             d_2D_path=base_path + 'davis_d_2D_features.npy',
                             d_3D_path=base_path + 'davis_d_3d_feature.npy',
                             label_file=base_path + 'davis_total_cid_unid.csv',
                             metadata_path=base_path + 'davis_total_cid_unid.csv')

# Reproduce 8:1:1 split with seed=3
shuffled_indices = np.random.permutation(len(data_list))
test_idx = shuffled_indices[int(0.9 * len(data_list)):]
test_loader = DataLoader(data_list, batch_size=cfg.SOLVER.BATCH_SIZE, drop_last=False, sampler=SubsetRandomSampler(test_idx))
print(f"Test set size: {len(test_idx)}")

# 4. Build model and load checkpoint
print("Building model...")
model = LightAttention(cfg).to(device)

# Update to the checkpoint path from your best run
model_path = "/content/EviDTI-main/runs/la_test1_05-04_02-44-43/checkpoint.pth"
print(f"Loading checkpoint: {model_path}")

if os.path.exists(model_path):
    # PyTorch 2.6 safe load
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)

    if isinstance(checkpoint, dict):
        if 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
        else:
            model.load_state_dict(checkpoint)
    elif isinstance(checkpoint, torch.nn.Module):
        model.load_state_dict(checkpoint.state_dict())

    print("Checkpoint loaded.")
else:
    print(f"Checkpoint not found: {model_path}")

print("\nPhase 1: Running official evaluation...")
model.eval()
solver = Solver(model, cfg, device)

try:
    # Phase 1: official evaluation (macro metrics)
    solver.evaluation(test_loader)
    print("Phase 1 complete.")

    # Phase 2: extract per-sample probability and uncertainty
    print("\nPhase 2: Extracting predictions and uncertainties...")
    model.eval()
    results_list = []

    with torch.no_grad():
        for batch in test_loader:

            # Move metadata to GPU manually
            metadata = batch.metadata
            sequence_lengths = metadata['length'][:, None].to(device)

            # Build attention mask
            mask = torch.arange(metadata['length'].max())[None, :] < metadata['length'][:, None]
            mask = mask.to(device)

            # Forward pass
            outputs = model(batch, mask=mask, sequence_lengths=sequence_lengths)

            # Extract labels and IDs
            labels = batch.l.cpu().numpy()
            cids = batch.cid
            unids = batch.unid

            # Compute probability and evidential uncertainty
            S = torch.sum(outputs, dim=1, keepdim=True)
            probs = (outputs[:, 1:2] / S).cpu().numpy().flatten()
            uncertainties = (2.0 / S).cpu().numpy().flatten()

            # Append to results
            for i in range(len(labels)):
                results_list.append({
                    'Drug_CID': cids[i],
                    'Target_Uniprot': unids[i],
                    'True_Label': labels[i],
                    'Predicted_Probability': probs[i],
                    'Uncertainty': uncertainties[i]
                })

    # Export predictions to CSV
    import pandas as pd
    df_results = pd.DataFrame(results_list)
    output_csv_path = 'EviDTI_Davis_Test_Predictions_Detailed.csv'
    df_results.to_csv(output_csv_path, index=False)

    print(f"Saved {len(df_results)} predictions to {output_csv_path}")
    print("\nPreview:")
    print(df_results.head())

except Exception as e:
    print(f"Error during evaluation: {e}")